**Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Is the stock market overvalued? Part 2: charting with Altair

In the lecture practical, we cleaned Shiller's data and saved `data_shiller_pe.csv`. Now we chart it.

**Altair** is a Python package that writes **Vega-Lite** for us, the same charting language you used in Weeks 1 and 2. Anything we make here can be embedded on your website.

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.
>
> Run a cell by clicking it and pressing **Shift + Enter**.

In [ ]:
import pandas as pd
import altair as alt

## 1. Load the data

Paste the **Raw** link to the CSV you uploaded to GitHub after the lecture.

In [ ]:
# Replace this with YOUR raw link (keep the quotes)
data_url = "https://raw.githubusercontent.com/jhellingsdata/jhellingsdata.github.io/refs/heads/main/Data/data_shiller_pe.csv"

df = pd.read_csv(data_url)
df.head()

> **Catch-up:** if you don't have your own file yet, leave the link above as it is. It points to a copy of the cleaned data.

---

## 2. A first chart

An Altair chart has three parts, the same as Vega-Lite:

| Altair | Vega-Lite | Meaning |
|---|---|---|
| `alt.Chart(df)` | `"data"` | Which data |
| `.mark_line()` | `"mark": "line"` | What shape to draw |
| `.encode(x=..., y=...)` | `"encoding"` | Which columns go where |

The letter after the colon is the **type**: `:T` for time (temporal), `:Q` for numbers (quantitative).

In [ ]:
chart = alt.Chart(df).mark_line().encode(
    x='Date_iso:T',
    y='PE_ratio:Q'
)

chart

### Under the hood: it's Vega-Lite

Altair has written a Vega-Lite spec for us. `.to_dict()` gives it to us as a Python dictionary, so we can look inside with keys:

In [ ]:
spec = chart.to_dict()

spec['mark']

In [ ]:
spec['encoding']

Recognise it? `mark`, `encoding`, `field`, `type`: exactly what you edited by hand in Week 2.

### What's that spike?

In 2009, during the financial crisis, company **earnings collapsed** almost to zero. Dividing by a tiny number gives a huge P/E, even though share prices *fell*.

This is one reason economists prefer a smoothed measure (see the CAPE stretch exercise at the end). It's also a good reminder to ask what's driving a pattern before you write a headline about it.

---

## 3. Style it

The same styling ideas as Week 2, written in Python. Change one thing at a time and re-run the cell.

| Week 2 (JSON) | Altair |
|---|---|
| `"title": {...}` | `alt.Chart(df, title=alt.Title(...))` |
| `"mark": {"color": ...}` | `.mark_line(color=...)` |
| `"axis": {"title": null}` | `alt.X(...).title(None)` |
| `"axis": {"grid": false}` | `alt.X(...).axis(grid=False)` |
| `"width": 400` | `.properties(width=400)` |

In [ ]:
chart_styled = alt.Chart(df, title=alt.Title(
    text='US stock prices are high relative to earnings',
    subtitle='Price-to-earnings ratio, S&P composite. Source: Robert Shiller',
    anchor='start'
)).mark_line(
    color='#c8102e',
    strokeWidth=1.5
).encode(
    x=alt.X('Date_iso:T').title(None).axis(grid=False),
    y=alt.Y('PE_ratio:Q').title(None)
).properties(
    width=500,
    height=280
)

chart_styled

### <font color='green'>Your turn</font>

Change at least **3** things in the chart above, e.g. the colour, line width, title, size, or font size of the axis labels (`.axis(labelFontSize=12)`).

---

## 4. Answer the question: add the long-run average

Is today's P/E *high*? Only compared with something. Let's add a dashed line at the long-run average.

Altair calculates the average for us with `mean(...)`, and `+` layers one chart on top of another.

In [ ]:
average_line = alt.Chart(df).mark_rule(
    color='grey',
    strokeDash=[4, 4]
).encode(
    y='mean(PE_ratio):Q'
)

chart_styled + average_line

Now the chart shows the comparison the title talks about: the P/E today is well above its long-run average.

> **Note:** this is a simple P/E ratio. The famous "Shiller P/E" (or CAPE) divides by the average of the last 10 years of earnings, to smooth out booms and recessions. See the stretch exercise below.

---

## 5. Save the chart for your website

So far, Altair has copied all the data **into** the chart. For a website, it's better to **link** to the data instead, just like in Week 2 (`"data": {"url": ...}`). The chart file stays small, and it updates when the data does.

We do this by giving `alt.Chart()` the **URL** instead of the table. With a URL, Altair can't see the columns, so the types (`:T`, `:Q`) are required.

In [ ]:
web_chart = alt.Chart(data_url, title=alt.Title(
    text='US stock prices are high relative to earnings',
    subtitle='Price-to-earnings ratio, S&P composite. Source: Robert Shiller',
    anchor='start'
)).mark_line(color='#c8102e', strokeWidth=1.5).encode(
    x=alt.X('Date_iso:T').title(None).axis(grid=False),
    y=alt.Y('PE_ratio:Q').title(None)
).properties(width=500, height=280)

web_average = alt.Chart(data_url).mark_rule(color='grey', strokeDash=[4, 4]).encode(y='mean(PE_ratio):Q')

web_final = web_chart + web_average
web_final

Save it as a JSON file:

In [ ]:
web_final.save('pe_ratio_chart.json')

In [ ]:
# Check: the data is a link, not thousands of rows
print(web_final.to_json()[:400])

In [ ]:
try:
    from google.colab import files
    files.download('pe_ratio_chart.json')
except ImportError:
    print('Not running in Colab: the file is saved next to this notebook.')

### Add it to your portfolio

1. Put `pe_ratio_chart.json` in your website folder.
2. In `portfolio.html`, add a new `<figure id="...">` and a `vegaEmbed` line pointing to the file (the same pattern as Weeks 1 and 2).
3. Check it with Live Server, then publish it.

---

## 6. Optional: save as an image

For reports and slides, you may want a PNG. Altair needs one extra package for this, which we install first:

In [ ]:
%pip install vl-convert-python --quiet

In [ ]:
web_final.save('pe_ratio_chart.png', scale_factor=2)   # scale_factor=2 for a sharper image

---

## <font color='green'>Stretch exercises</font>

**1. The real "Shiller P/E" (CAPE).** Using the full dataset from the lecture notebook, calculate:

```python
df['CAPE'] = df['Price_real'] / df['Earnings_real'].rolling(120).mean()
```

`.rolling(120).mean()` is the average of the last 120 months (10 years). Chart it. How does it compare with the simple P/E?

**2. Highlight a period.** Look up `mark_rect` in the [Altair gallery](https://altair-viz.github.io/gallery/index.html) and shade the 2000 dot-com bubble.

**3. More Altair.** Work through **DS3_3_AltairBasics**.

In [ ]:
# Your code here
